In [2]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

import os
from contextlib import asynccontextmanager

In [8]:
MCP_URL = os.environ.get("MCP_URL", "https://order-mcp-74afyau24q-uc.a.run.app/mcp")


@asynccontextmanager
async def mcp_session():
    async with streamable_http_client(MCP_URL) as (read, write, _get_session_id):
        async with ClientSession(read, write) as session:
            await session.initialize()
            yield session

In [14]:
async def _list_tools():
    async with mcp_session() as session:
        return await session.list_tools()


tools_result = await _list_tools()
print(f"{len(tools_result.tools)} tool(s) on {MCP_URL!r}:\n")
for t in tools_result.tools:
    print(f"• {t.name}")
    if t.description:
        print(f"  {t.description}")
    props = (t.inputSchema or {}).get("properties") or {}
    if props:
        print(f"  parameters: {', '.join(props.keys())}")
    print()

8 tool(s) on 'https://order-mcp-74afyau24q-uc.a.run.app/mcp':

• list_products
  List products with optional filters.

    Args:
        category: Filter by category (e.g., "Computers", "Monitors")
        is_active: Filter by active status (True/False)

    Returns:
        Formatted string with products, one per line

    Use cases:
        - Browse inventory by category
        - Check stock levels
        - Find available products
    
  parameters: category, is_active

• get_product
  Get detailed product information by SKU.

    Args:
        sku: Product SKU (e.g., "COM-0001")

    Returns:
        Formatted product details

    Raises:
        ProductNotFoundError: If SKU doesn't exist

    Use cases:
        - Get current price
        - Check inventory for specific item
        - Verify product details before ordering
    
  parameters: sku

• search_products
  Search products by name or description.

    Args:
        query: Search term (case-insensitive, partial match)

   

In [13]:
TOOL_NAME = os.environ.get("MCP_TOOL_NAME", "auth")
# Optional JSON object, e.g. '{"order_id":"123"}'
TOOL_ARGUMENTS: dict = {}


async def _call_tool():
    async with mcp_session() as session:
        return await session.call_tool(TOOL_NAME, TOOL_ARGUMENTS)


out = await _call_tool()
print(out)

meta=None content=[TextContent(type='text', text='Unknown tool: auth', annotations=None, meta=None)] structuredContent=None isError=True
